Input: 
1. data: pd.DataFrame, with PV generation data and weather data, no normalized data

Output: 
1. The WIS graph
2. The table of 19 confidence intervals results, which consists of sharpness, calibration, interval score, and WIS.


This file splits the households into 2 groups, group 1 with 10 households as training data and group 2 with 11 households as testing.

1. Create 
2. Run capacity estimation with group 1 and group 2, obtain the error factor
3. Train point regressor with group 1.
4. Make point predictions with group 2
5. Construct the prediction interval for group 2 
6. Evaluate the intervals

In [1]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')
# Get the parent directory
parent_folder = os.path.dirname(os.getcwd())
# add the parent directory to the Python path so that the scripts can be imported
sys.path.append(parent_folder)

In [2]:
# Imports
import timeit
from modules.preprocessing import data_preprocessor
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
from lightgbm import LGBMRegressor
from mapie.regression import MapieQuantileRegressor
from crepes.extras import margin, DifficultyEstimator, MondrianCategorizer
from crepes import WrapRegressor
import scienceplots
from modules.capacity_estimation_aus import (
    capacity_estimation_base_load,
    sensitivity_analysis_irradiance_threshold,
)

from modules.utils import (
    get_values,
    calc_penalty,
    calculate_coverage,
    calculate_mean_size,
    calculate_median_size,
    calc_multi_level,
    calc_multi_level_WIS,
    calc_multi_level_cqr,
    calc_multi_level_cqr_WIS,
    calc_size_stratified_coverage,
    plot_actual_vs_predicted,
    calc_percentage_error,
    calc_mean_percentage_error,
    calc_error_factor,
    calc_r2_score,
    read_column_list_from_config,
    split_households,
    preprocess_two_groups,
    cp_data_preparation,
    cqr_multi_level_eval,
    adaptive_mondrian_bins
)
from modules.capacity_estimation_aus import capacity_estimation_error_factor
from modules.qrnn import QRNN

In [3]:
# constants
general_seed = 42
my_no_bins = 6
n_splits = 10
confidences = [i / 100 for i in range(5, 100, 5)]
list_of_features = ["ghi_norm", "total_net_norm", "air_temp_norm",
                    "azimuth_norm"]
prob_methods = ["qr", "cqr", "qrnn", "cp", "cps",
                "cp_mond", "cps_mond", "cp_amb", "cps_amb"]
cap = 1.1
plt.style.use(["science"])

In [4]:
metrics = ["sharpness", "calibration",
           "interval_score", "wis", 'time_total', 'time_prob']
all_metrics_test = np.zeros((n_splits, len(prob_methods), len(metrics)))
all_metrics_cal = np.zeros((n_splits, len(prob_methods), len(metrics)))

In [5]:
# Read data
fname = "../data/ausgrid/ausgrid_merged_unprocessed.csv"
df = pd.read_csv(fname)
df = df[(df['datetime'] >= '2012-07-01')
        & (df['datetime'] <= '2013-07-01')]
# reset the index
df = df.reset_index(drop=True)

In [6]:
warnings.filterwarnings('ignore')
for split in range(n_splits):
    start_time = timeit.default_timer()
    # Set a different seed for each split
    seed = split + 1  # using split+1 to avoid seed=0
    np.random.seed(seed)
    print(f"Split {split + 1} of {n_splits}")
    time_columns = read_column_list_from_config("../config/time_columns.txt")
    weather_columns = read_column_list_from_config(
        "../config/weather_columns_preprocess.txt"
    )
    feature_columns = read_column_list_from_config(
        "../config/feature_columns_preprocess.txt"
    )
    # Split the households into 2 groups
    n_1, n_2 = 39, 38
    df_1, df_2 = split_households(
        df,
        n_1=n_1,
        n_2=n_2,
        seed=seed,
        date_columns=time_columns,
        weather_columns=weather_columns,
    )
    preprocessor_1 = data_preprocessor(df_1)
    preprocessor_2 = data_preprocessor(df_2)
    # Preprocess the data
    preprocessed_data_part1, preprocessed_data_part2 = preprocess_two_groups(
        preprocessor_1=preprocessor_1,
        preprocessor_2=preprocessor_2,
        feature_columns=feature_columns,
    )
    # Capacity Estimation
    capacity_error_factor = capacity_estimation_error_factor(
        df_1, df_2)
    # prepare data for CP
    X_prop_train, y_prop_train, X_cal, y_cal, X_test, y_test, len_test = (
        cp_data_preparation(
            df_1=preprocessed_data_part1,
            df_2=preprocessed_data_part2,
            list_of_features=list_of_features,
        )
    )
    y_test = y_test * capacity_error_factor
    # time it
    elapsed_1 = timeit.default_timer() - start_time
    # run the 8 prob methods
    for i, prob_method in enumerate(prob_methods):
        start_time_2 = timeit.default_timer()
        if prob_method == "qr" or prob_method == "cqr":
            sharpness, calibration_error, interval_score, wis = cqr_multi_level_eval(
                X_test=X_test,
                y_test=y_test,
                confidences=confidences,
                X_prop_train=X_prop_train,
                y_prop_train=y_prop_train,
                X_cal=X_cal,
                y_cal=y_cal,
                calibrate=True if prob_method == "cqr" else False,
            )
        elif prob_method == "cp" or prob_method == "cps":
            lgb = WrapRegressor(
                LGBMRegressor(
                    n_jobs=-1, n_estimators=100, random_state=seed, verbose=-1
                )
            )
            lgb.fit(X_prop_train, y_prop_train)
            learner_prop = lgb.learner
            learner = WrapRegressor(learner_prop)
            learner.calibrate(
                X_cal, y_cal, cps=True if prob_method == "cps" else False, seed=seed
            )
            sharpness, calibration_error, interval_score = calc_multi_level(
                learner, X_test, y_test
            )
            wis = calc_multi_level_WIS(learner, X_test, y_test)
        elif prob_method == "cp_mond" or prob_method == "cps_mond":
            learner_prop = lgb.learner
            learner = WrapRegressor(learner_prop)
            mc = MondrianCategorizer()
            mc.fit(X=X_cal, f=get_values, no_bins=my_no_bins)
            learner.calibrate(
                X_cal,
                y_cal,
                cps=True if prob_method == "cps_mond" else False,
                mc=mc,
                seed=seed,
            )
            sharpness, calibration_error, interval_score = calc_multi_level(
                learner, X_test, y_test
            )
            wis = calc_multi_level_WIS(learner, X_test, y_test)
        elif prob_method == "cp_amb" or prob_method == "cps_amb":
            learner_prop = lgb.learner
            learner = WrapRegressor(learner_prop)
            optimal_no_bins = adaptive_mondrian_bins(
                learner, X_cal, y_cal, seed, prob_method)
            mc = MondrianCategorizer()
            mc.fit(X=X_cal, f=get_values, no_bins=optimal_no_bins)
            learner.calibrate(
                X_cal,
                y_cal,
                cps=True if prob_method == "cps_amb" else False,
                mc=mc,
                seed=seed,
            )
            sharpness, calibration_error, interval_score = calc_multi_level(
                learner, X_test, y_test
            )
            wis = calc_multi_level_WIS(learner, X_test, y_test)
        elif prob_method == "qrnn":
            X_train = np.concatenate([X_prop_train, X_cal], axis=0)
            y_train = np.concatenate([y_prop_train, y_cal], axis=0)
            qrnn = QRNN(random_state=seed, hidden_size=64, n_hidden_layers=2,
                        epochs=150, batch_size=32, lr=1e-3)
            qrnn.fit(X_train, y_train)
            qrnn_quantiles = np.clip(qrnn.predict(X_test), 0, cap)
            q_levels = qrnn.quantiles
            conf_q = [c for c in confidences if np.any(np.isclose(q_levels, (1 - c) / 2))
                      and np.any(np.isclose(q_levels, (1 + c) / 2))]

            def _qi(levels, q): return int(np.argmin(np.abs(levels - q)))
            mean_sz, pen = [], []
            for c in conf_q:
                lo, hi = _qi(q_levels, (1 - c) / 2), _qi(q_levels, (1 + c) / 2)
                lower, upper = qrnn_quantiles[:, lo], qrnn_quantiles[:, hi]
                mean_sz.append(calculate_mean_size(
                    np.stack([lower, upper], axis=1)))
                pen.append(calc_penalty(y_test, lower, upper, c))
            sharpness = np.mean(mean_sz)
            calibration_error = np.mean(pen)
            interval_score = sharpness + calibration_error
            med = qrnn_quantiles[:, _qi(q_levels, 0.5)]
            K = len(conf_q)
            w0 = 0.5
            wk = np.zeros((len(y_test), K))
            for idx, c in enumerate(conf_q):
                lo, hi = _qi(q_levels, (1 - c) / 2), _qi(q_levels, (1 + c) / 2)
                lower, upper = qrnn_quantiles[:, lo], qrnn_quantiles[:, hi]
                wd = upper - lower
                alpha_i = 1 - c
                p = np.zeros_like(y_test)
                p[y_test < lower] = 2 * \
                    (lower[y_test < lower] - y_test[y_test < lower]) / alpha_i
                p[y_test > upper] = 2 * \
                    (y_test[y_test > upper] - upper[y_test > upper]) / alpha_i
                wk[:, idx] = (alpha_i / 2) * (wd + p)
            wis = np.mean((1 / (K + 0.5)) *
                          (w0 * np.abs(y_test - med) + np.sum(wk, axis=1)))
        elapsed_2 = timeit.default_timer() - start_time_2
        elapsed_total = elapsed_2 + elapsed_1
        all_metrics_test[split, i, 0] = sharpness
        all_metrics_test[split, i, 1] = calibration_error
        all_metrics_test[split, i, 2] = interval_score
        all_metrics_test[split, i, 3] = wis
        all_metrics_test[split, i, 4] = elapsed_total
        all_metrics_test[split, i, 5] = elapsed_2

Split 1 of 10
Split 2 of 10
Split 3 of 10
Split 4 of 10
Split 5 of 10
Split 6 of 10
Split 7 of 10
Split 8 of 10
Split 9 of 10
Split 10 of 10


In [7]:
# save all_metrics_test to a numpy file
np.save(
    f"../data/results/all_metrics_test_ausgrid_seed{general_seed}_factor_1year_amb.npy",
    all_metrics_test
)

In [8]:
# give me the statistics of all metrics for each method, organized by the order of prob_methods
all_metrics_test = np.load(
    f"../data/results/all_metrics_test_ausgrid_seed{general_seed}_factor_1year_amb.npy"
)
# Calculate statistics across all splits for each method and metric
results = []
for i, method in enumerate(prob_methods):
    for j, metric in enumerate(metrics):
        values = all_metrics_test[:, i, j]
        results.append({
            'Method': method,
            'Metric': metric,
            'Mean': np.mean(values),
            'Std': np.std(values),
            'Min': np.min(values),
            'Max': np.max(values),
            'Median': np.median(values),
        })

# Create DataFrame
df_stats = pd.DataFrame(results)

# Display results
print("\n=== Statistics across all splits ===\n")
print(df_stats.to_string(index=False))

# Create pivot table with mean ± std format
print("\n=== Mean ± Std by method ===\n")

# Create a pivot table for mean and std separately
df_pivot_mean = df_stats.pivot(index='Method', columns='Metric', values='Mean')
df_pivot_std = df_stats.pivot(index='Method', columns='Metric', values='Std')

# Reorder to match prob_methods order
df_pivot_mean = df_pivot_mean.reindex(prob_methods)
df_pivot_std = df_pivot_std.reindex(prob_methods)

# Reorder columns to match metrics order
df_pivot_mean = df_pivot_mean.reindex(columns=metrics)
df_pivot_std = df_pivot_std.reindex(columns=metrics)

# Format as mean ± std (as percentage of mean)
df_mean_std = pd.DataFrame(index=df_pivot_mean.index,
                           columns=df_pivot_mean.columns)
for idx in df_pivot_mean.index:
    for col in df_pivot_mean.columns:
        mean_val = df_pivot_mean.loc[idx, col]
        std_val = df_pivot_std.loc[idx, col]
        std_pct = (std_val / mean_val * 100) if mean_val != 0 else 0
        df_mean_std.loc[idx, col] = f"{mean_val:.4f} ± {std_pct:.1f}%"

print(df_mean_std)


=== Statistics across all splits ===

  Method         Metric      Mean      Std       Min       Max    Median
      qr      sharpness  0.067644 0.005342  0.060781  0.078166  0.067233
      qr    calibration  0.178269 0.094896  0.072621  0.351966  0.131976
      qr interval_score  0.245914 0.094549  0.144830  0.417388  0.201590
      qr            wis  0.048444 0.016775  0.030805  0.079926  0.040850
      qr     time_total 15.481055 0.126311 15.203774 15.633545 15.473053
      qr      time_prob  3.737851 0.031265  3.658246  3.768609  3.744635
     cqr      sharpness  0.087870 0.009160  0.076958  0.106251  0.085686
     cqr    calibration  0.144663 0.090238  0.059391  0.306710  0.095659
     cqr interval_score  0.232533 0.085609  0.145783  0.388692  0.191393
     cqr            wis  0.047167 0.015994  0.030758  0.077727  0.040013
     cqr     time_total 15.654083 0.123516 15.397846 15.796156 15.661213
     cqr      time_prob  3.910880 0.041736  3.823993  3.995615  3.912703
    qrnn    

In [9]:
df_mean_std

Metric,sharpness,calibration,interval_score,wis,time_total,time_prob
Method,,,,,,
qr,0.0676 ± 7.9%,0.1783 ± 53.2%,0.2459 ± 38.4%,0.0484 ± 34.6%,15.4811 ± 0.8%,3.7379 ± 0.8%
cqr,0.0879 ± 10.4%,0.1447 ± 62.4%,0.2325 ± 36.8%,0.0472 ± 33.9%,15.6541 ± 0.8%,3.9109 ± 1.1%
qrnn,0.0655 ± 9.6%,0.1519 ± 51.5%,0.2174 ± 34.2%,0.0445 ± 29.5%,29.5476 ± 0.6%,17.8044 ± 1.0%
cp,0.0871 ± 13.4%,0.1578 ± 62.3%,0.2449 ± 38.2%,0.0492 ± 35.2%,11.8408 ± 1.0%,0.0976 ± 10.9%
cps,0.0714 ± 5.1%,0.1495 ± 48.1%,0.2208 ± 31.8%,0.0442 ± 29.4%,11.9254 ± 1.0%,0.1822 ± 6.4%
cp_mond,0.0931 ± 14.0%,0.1443 ± 64.2%,0.2374 ± 36.7%,0.0483 ± 34.2%,11.8194 ± 1.0%,0.0762 ± 18.2%
cps_mond,0.0733 ± 2.8%,0.1456 ± 44.6%,0.2189 ± 29.2%,0.0443 ± 26.7%,11.8919 ± 1.0%,0.1487 ± 2.9%
cp_amb,0.1013 ± 13.5%,0.1345 ± 64.1%,0.2358 ± 34.2%,0.0482 ± 33.0%,12.5160 ± 1.0%,0.7728 ± 3.2%
cps_amb,0.0797 ± 2.5%,0.1359 ± 44.6%,0.2156 ± 27.7%,0.0440 ± 26.0%,13.1735 ± 0.9%,1.4303 ± 1.3%
